# NeuroCare on Kaggle: the full stack with a public link

Runs the fine-tuned **Aura** model (Llama 3.2 3B, Ollama on Kaggle's free GPU), the NeuroCare server and the website,
and prints a public **https://…trycloudflare.com** link that works on phones, with voice and camera.

**Before running (one time):**
1. Kaggle → **Datasets → New Dataset** → upload `aura-empathy-llama-3.2-3b.Q4_K_M.gguf` → keep it **Private** → Create.
2. In this notebook: **Add Input** (right panel) → *Your Work / Datasets* → add that dataset.
3. Right panel **Session options**: *Accelerator* = **GPU T4 x2**, *Internet* = **On**
   (Internet needs a phone-verified Kaggle account: Settings → Phone verification).

**Run it:**
- *Laptop can be switched off*: **Save Version → Save & Run All (Commit)**. Open the running version → **Logs**; the link appears
  after about 5 minutes and stays live for up to 12 hours.
- *Quick test*: **Run All** here and keep this tab open.

The link changes on every run. To get it on your phone automatically, set `NTFY_TOPIC` below to any long random word
and open `https://ntfy.sh/<that word>` (free, no account).

In [ ]:
REPO = "https://github.com/asmitkundu1004-source/NeuroCare.git"
NTFY_TOPIC = ""          # optional: e.g. "neurocare-demo-7f3k9q"; the link is pushed to https://ntfy.sh/<topic>
MAX_HOURS = 11.5         # Kaggle stops sessions at 12 hours

In [ ]:
import glob, os, re, subprocess, sys, time, urllib.request

W = "/kaggle/working"
os.chdir(W)

def sh(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

def up(url, timeout=2):
    try:
        urllib.request.urlopen(url, timeout=timeout)
        return True
    except Exception:
        return False

def wait(url, seconds, what):
    for _ in range(seconds):
        if up(url):
            return
        time.sleep(1)
    raise RuntimeError(f"{what} did not start; see the log files in {W}")

ggufs = glob.glob("/kaggle/input/**/*.gguf", recursive=True)
assert ggufs, "Model not found. Add your Aura model dataset with 'Add Input' (see the first cell)."
GGUF = ggufs[0]
print("Model:", GGUF)
sh("nvidia-smi --query-gpu=name,memory.total --format=csv || true")

# Code
if os.path.isdir("NeuroCare/.git"):
    sh("git -C NeuroCare pull --ff-only")
else:
    sh(f"git clone --depth 1 {REPO} NeuroCare")
ROOT = f"{W}/NeuroCare"

# Tools: Ollama, Node.js 20, cloudflared
sh("apt-get -qq update && apt-get -qq install -y zstd > /dev/null")
if not os.path.exists("/usr/local/bin/ollama"):
    sh("curl -fsSL https://ollama.com/install.sh | sh")
if not os.path.exists(f"{W}/node/bin/node"):
    sums = urllib.request.urlopen("https://nodejs.org/dist/latest-v20.x/SHASUMS256.txt").read().decode()
    tarball = re.search(r"node-v20[\d.]+-linux-x64\.tar\.xz", sums).group(0)
    sh(f"curl -fsSL https://nodejs.org/dist/latest-v20.x/{tarball} | tar -xJ -C {W} && mv {W}/{tarball[:-7]} {W}/node")
os.environ["PATH"] = f"{W}/node/bin:" + os.environ["PATH"]
if not os.path.exists(f"{W}/cloudflared"):
    sh(f"curl -fsSL -o {W}/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x {W}/cloudflared")

# Language model on the GPU, kept loaded between messages
env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", OLLAMA_KEEP_ALIVE="-1")
if not up("http://127.0.0.1:11434/api/tags"):
    ollama = subprocess.Popen(["ollama", "serve"], env=env, stdout=open(f"{W}/ollama.log", "w"), stderr=subprocess.STDOUT)
    wait("http://127.0.0.1:11434/api/tags", 60, "Ollama")
modelfile = open(f"{ROOT}/models/aura-llm/Modelfile").read()
open(f"{W}/Modelfile", "w").write(re.sub(r"^FROM .*$", f"FROM {GGUF}", modelfile, count=1, flags=re.M))
sh(f"ollama create aura -f {W}/Modelfile")
sh('ollama run aura "Say hello in five words." || true')   # loads it onto the GPU

# Server and website
sh(f"{sys.executable} -m pip install -q -r {ROOT}/backend/requirements.txt")
sh(f"cd {ROOT}/web && npm ci --no-audit --no-fund --loglevel=error && npm run build")
print("Setup finished.")

In [ ]:
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "neurocare.server:app", "--host", "127.0.0.1", "--port", "8000", "--proxy-headers"],
    cwd=f"{ROOT}/backend", env=dict(os.environ, NEUROCARE_TTS="0"),
    stdout=open(f"{W}/server.log", "w"), stderr=subprocess.STDOUT,
)
wait("http://127.0.0.1:8000/api/health", 120, "The NeuroCare server")
print("Server:", urllib.request.urlopen("http://127.0.0.1:8000/api/health").read().decode()[:300])

tunnel = subprocess.Popen(
    [f"{W}/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
    stdout=open(f"{W}/tunnel.log", "w"), stderr=subprocess.STDOUT,
)
URL = None
for _ in range(90):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(f"{W}/tunnel.log").read())
    if m:
        URL = m.group(0)
        break
assert URL, f"No public link from Cloudflare; see {W}/tunnel.log"

banner = f"NeuroCare is live:  {URL}"
print("\n" + "=" * len(banner) + "\n" + banner + "\n" + "=" * len(banner) + "\n", flush=True)
if NTFY_TOPIC:
    try:
        urllib.request.urlopen(urllib.request.Request(f"https://ntfy.sh/{NTFY_TOPIC}", data=URL.encode(), headers={"Title": "NeuroCare is live"}), timeout=10)
        print(f"Link also sent to https://ntfy.sh/{NTFY_TOPIC}")
    except Exception as e:
        print("ntfy failed:", e)

# Keep the session alive and restart anything that stops.
start = time.time()
while time.time() - start < MAX_HOURS * 3600:
    time.sleep(60)
    if server.poll() is not None:
        print("Server stopped, restarting. Last log lines:\n", open(f"{W}/server.log").read()[-1500:], flush=True)
        server = subprocess.Popen(server.args, cwd=f"{ROOT}/backend", env=dict(os.environ, NEUROCARE_TTS="0"),
                                  stdout=open(f"{W}/server.log", "a"), stderr=subprocess.STDOUT)
    if tunnel.poll() is not None:
        print("The tunnel stopped; run this cell again for a new link.", flush=True)
        break
    mins = int((time.time() - start) / 60)
    if mins % 15 == 0:
        print(f"[{mins // 60}h{mins % 60:02d}] still live at {URL}", flush=True)
tunnel.terminate()
server.terminate()